In [ ]:
import os
import sys
import torch
import pandas as pd
import mlflow
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
from datetime import datetime
sys.path.append(os.path.abspath(os.path.join('..')))

from models import HybridModel
from utils import mol_to_graph, MLFlowManager, train_hybrid_model, evaluate_hybrid_model

def log_regression_plots(y_true, y_pred, run_name):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], '--r', lw=2)
    plt.xlabel("Actual pIC50")
    plt.ylabel("Predicted pIC50")
    plt.title(f"Regression Fit - {run_name}")
    plot_path = "pred_vs_actual.png"
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()

features = [
    'alogp', 'psa', 'hba', 'hbd', 'num_ro5_violations', 'qed_weighted',
    'logP_over_PSA', 'HBA_HBD_ratio', 'HBA_HBD_sum'
]
num_features = len(features)
parquet_path = "parquets/df_ml_with_scaffold.parquet"
df = pd.read_parquet(parquet_path)
dataset_path = "/home/pkuszn/repos/WSzI/src/notebooks/data/chembl_dataset.pt"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if os.path.exists(dataset_path):
    print("Loading existing dataset...")
    dataset = torch.load(dataset_path, weights_only=False)
else:
    print("Converting SMILES to graphs...")
    dataset = Parallel(n_jobs=-1)(
        delayed(mol_to_graph)(
            s, 
            y, 
            row[features].values.astype(float)
        ) 
        for s, y, row in tqdm(
            zip(df['canonical_smiles'], df['pic50'], [row for _, row in df.iterrows()]), 
            total=len(df), 
            desc="Converting"
        )
    )
    dataset = [d for d in dataset if d is not None]
    torch.save(dataset, dataset_path)

train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

model = HybridModel(num_node_features=4, num_extra_features=num_features, hidden_channels=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()
mf = MLFlowManager(experiment_name="ChEMBL_HybridModel_Scaffold_Split")

now = str(int(datetime.now().timestamp()))
run_name = f"Hybrid_Run_{now}"
with mlflow.start_run(run_name=run_name):
    print("Starting Training...")
    for epoch in range(50):
        loss = train_hybrid_model(model, train_loader, optimizer, device, mf, run_name)
        if epoch % 10 == 0:
            print(f"Epoch {epoch} complete, Loss: {loss:.4f}")
    
    r2, mae, y_true, y_pred = evaluate_hybrid_model(model, test_loader, device)
    
    mlflow.log_metrics({"test_r2": r2, "test_mae": mae})
    log_regression_plots(y_true, y_pred, "Hybrid_Final")
    print(f"Final Test: R2={r2:.4f}, MAE={mae:.4f}")

ModuleNotFoundError: No module named 'models'